# Day 071 — Exercise 5: ImageSearchEngine

**What you'll build:** `ImageSearchEngine` — the complete Vision RAG pipeline as a single reusable class.

**Why it matters:** The class is the deliverable that a developer drops into any app. One constructor call to configure, then `.add_image` to index and `.search` to query.

In [ ]:
import base64, hashlib, io
import numpy as np
from PIL import Image

_SEARCH_PROMPT = (
    'Describe this image in detail for use in a semantic search index. '
    'Include: main subjects, colors, textures, setting, and visible text. '
    'Write one concise paragraph of 2-3 sentences.'
)

def image_to_base64(img, format='PNG'):
    buf = io.BytesIO()
    out = img
    if format.upper() in ('JPEG', 'JPG') and img.mode in ('RGBA', 'P'):
        out = img.convert('RGB')
    out.save(buf, format=format)
    return base64.b64encode(buf.getvalue()).decode()

def cosine_similarity(a, b):
    va = np.array(a, dtype=np.float32)
    vb = np.array(b, dtype=np.float32)
    denom = float(np.linalg.norm(va) * np.linalg.norm(vb))
    if denom == 0.0:
        return 0.0
    return float(np.dot(va, vb) / denom)

class ImageIndex:
    def __init__(self):
        self._items = []
    def add(self, image_id, description, embedding, metadata=None):
        self._items.append({'id': image_id, 'description': description,
                            'embedding': np.array(embedding, dtype=np.float32),
                            'metadata': metadata or {}})
    def search(self, query_embedding, n=5):
        if not self._items:
            return []
        q = np.array(query_embedding, dtype=np.float32)
        scored = [(cosine_similarity(q, item['embedding']), item) for item in self._items]
        scored.sort(key=lambda x: -x[0])
        top = scored[:min(n, len(scored))]
        return [{'id': item['id'], 'description': item['description'],
                 'score': float(score), 'metadata': item['metadata']}
                for score, item in top]
    def __len__(self):
        return len(self._items)

def describe_image_for_search(img, describe_fn=None):
    img_b64 = image_to_base64(img)
    if describe_fn is not None:
        return describe_fn(img_b64, _SEARCH_PROMPT)
    import ollama
    resp = ollama.chat(model='llava',
                       messages=[{'role': 'user', 'content': _SEARCH_PROMPT, 'images': [img_b64]}])
    return resp['message']['content'].strip()

def embed_text(text, embed_fn=None):
    if embed_fn is not None:
        return embed_fn(text)
    import ollama
    resp = ollama.embeddings(model='nomic-embed-text', prompt=text)
    return resp['embedding']

def index_images(images_with_ids, describe_fn=None, embed_fn=None):
    index = ImageIndex()
    for image_id, img, metadata in images_with_ids:
        desc = describe_image_for_search(img, describe_fn=describe_fn)
        emb  = embed_text(desc, embed_fn=embed_fn)
        index.add(image_id, desc, emb, metadata or {})
    return index

def search_by_text(query, index, embed_fn=None, n=5):
    q_emb = embed_text(query, embed_fn=embed_fn)
    return index.search(q_emb, n=n)

def search_by_image(img, index, describe_fn=None, embed_fn=None, n=5):
    desc  = describe_image_for_search(img, describe_fn=describe_fn)
    q_emb = embed_text(desc, embed_fn=embed_fn)
    return index.search(q_emb, n=n)

import hashlib
def _mock_describe(img_b64, prompt):
    h = int(hashlib.md5(img_b64.encode()).hexdigest()[:4], 16)
    labels = ['a red apple on a table', 'a blue ocean wave',
              'a green forest path', 'a yellow sunflower field']
    return labels[h % len(labels)]

def _mock_embed(text):
    h = int(hashlib.md5(text.encode()).hexdigest()[:8], 16)
    return [((h >> (i * 8)) & 0xff) / 128.0 - 1.0 for i in range(4)]


## Task

Implement `ImageSearchEngine`:

- `__init__(describe_fn=None, embed_fn=None)`: store both, `self._index = ImageIndex()`
- `add_image(image_id, img, metadata=None) -> str`: describe → embed → add to index → return desc
- `add_batch(images_with_ids) -> list[str]`: `[self.add_image(iid, img, meta) for ...]`
- `search(query, n=5) -> list[dict]`: `search_by_text(query, self._index, embed_fn=self._embed_fn, n=n)`
- `search_by_image(img, n=5) -> list[dict]`: module-level `search_by_image` with both fns
- `__len__() -> int`: `len(self._index)`

## Your Implementation

In [ ]:
class ImageSearchEngine:
    """Content-based image search engine.

    Inject describe_fn and embed_fn for testing without Ollama.
    """

    def __init__(self, describe_fn=None, embed_fn=None) -> None:
        raise NotImplementedError

    def add_image(self, image_id: str, img, metadata=None) -> str:
        """Index one image. Returns the generated description."""
        raise NotImplementedError

    def add_batch(self, images_with_ids: list) -> list:
        """Index a batch of (image_id, img, metadata) tuples.

        Returns list of generated description strings.
        """
        raise NotImplementedError

    def search(self, query: str, n: int = 5) -> list:
        """Search by text query. Returns list of result dicts."""
        raise NotImplementedError

    def search_by_image(self, img, n: int = 5) -> list:
        """Search by image query. Returns list of result dicts."""
        raise NotImplementedError

    def __len__(self) -> int:
        raise NotImplementedError


In [ ]:
class ImageSearchEngine:
    def __init__(self, describe_fn=None, embed_fn=None):
        self._describe_fn = describe_fn
        self._embed_fn    = embed_fn
        self._index       = ImageIndex()

    def add_image(self, image_id, img, metadata=None):
        desc = describe_image_for_search(img, describe_fn=self._describe_fn)
        emb  = embed_text(desc, embed_fn=self._embed_fn)
        self._index.add(image_id, desc, emb, metadata or {})
        return desc

    def add_batch(self, images_with_ids):
        return [self.add_image(iid, img, meta)
                for iid, img, meta in images_with_ids]

    def search(self, query, n=5):
        return search_by_text(query, self._index,
                              embed_fn=self._embed_fn, n=n)

    def search_by_image(self, img, n=5):
        return search_by_image(img, self._index,
                               describe_fn=self._describe_fn,
                               embed_fn=self._embed_fn, n=n)

    def __len__(self):
        return len(self._index)


## Automated checks

In [ ]:
score, total = 0, 5
try:
    engine = ImageSearchEngine(describe_fn=_mock_describe, embed_fn=_mock_embed)
    assert len(engine) == 0
    score += 1; print("\u2705 empty engine has len 0")

    # add_image returns a description string
    img = Image.new('RGB', (16, 16), (220, 50, 50))
    desc = engine.add_image('img1', img, {'tag': 'red'})
    assert isinstance(desc, str) and len(desc) > 0
    assert len(engine) == 1
    score += 1; print("\u2705 add_image returns description, __len__ increments")

    # add_batch indexes multiple images
    batch = [
        ('img2', Image.new('RGB', (16,16), (50, 100, 220)), {'tag': 'blue'}),
        ('img3', Image.new('RGB', (16,16), (50, 180, 80)),  {'tag': 'green'}),
    ]
    descs = engine.add_batch(batch)
    assert len(descs) == 2 and all(isinstance(d, str) for d in descs)
    assert len(engine) == 3
    score += 1; print("\u2705 add_batch adds all items, returns descriptions")

    # search returns sorted results with correct keys
    results = engine.search('a red object', n=2)
    assert len(results) <= 2
    assert all('id' in r and 'score' in r and 'metadata' in r for r in results)
    if len(results) >= 2:
        assert results[0]['score'] >= results[1]['score']
    score += 1; print("\u2705 search returns sorted results with correct keys")

    # search_by_image returns results
    q_img = Image.new('RGB', (16, 16), (200, 100, 100))
    img_results = engine.search_by_image(q_img, n=2)
    assert isinstance(img_results, list) and len(img_results) <= 2
    score += 1; print("\u2705 search_by_image returns list of result dicts")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
class ImageSearchEngine:
    def __init__(self, describe_fn=None, embed_fn=None):
        self._describe_fn = describe_fn
        self._embed_fn    = embed_fn
        self._index       = ImageIndex()

    def add_image(self, image_id, img, metadata=None):
        desc = describe_image_for_search(img, describe_fn=self._describe_fn)
        emb  = embed_text(desc, embed_fn=self._embed_fn)
        self._index.add(image_id, desc, emb, metadata or {})
        return desc

    def add_batch(self, images_with_ids):
        return [self.add_image(iid, img, meta)
                for iid, img, meta in images_with_ids]

    def search(self, query, n=5):
        return search_by_text(query, self._index,
                              embed_fn=self._embed_fn, n=n)

    def search_by_image(self, img, n=5):
        return search_by_image(img, self._index,
                               describe_fn=self._describe_fn,
                               embed_fn=self._embed_fn, n=n)

    def __len__(self):
        return len(self._index)
```

**Why does `search_by_image` call the module-level function rather than being re-implemented?** The module-level function is already tested. Calling it avoids duplicate logic. The class method only adds the `self._describe_fn` and `self._embed_fn` arguments — that's the entire reason the class method exists.

</details>